In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from groq import Groq
from db.qdrant_setup import COLLECTION_NAME
from dotenv import load_dotenv
import qdrant_client as qc
from rank_bm25 import BM25Okapi
from collections import defaultdict
from sentence_transformers import CrossEncoder
import re
from fastembed import SparseTextEmbedding


In [ ]:
load_dotenv() 

qdrant = QdrantClient(path="./qdrant_storage")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
groq_client  = Groq(api_key=os.getenv("GROQ_API_KEY"))
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")


In [ ]:
# ── State ─────────────────────────────────────────────────────────
class RAGState(TypedDict):
    question: str
    query_vector: List[float]
    retrieved_chunks: List[dict]
    relevance_scores: List[str]     # "relevant" | "irrelevant" per chunk
    rewrite_count: int              # prevent infinite loops
    prompt: str
    answer: str
    hallucination_check: str  
    dense_chunks: List[dict]
    sparse_chunks: List[dict]
    retrieved_chunks: List[dict]

In [ ]:
# ── Node 1: Embed the question ────────────────────────────────────
def embed_node(state: RAGState):
    print(f"\n🔍 Embedding question: {state['question']}")

    vector = embed_model.encode(state["question"]).tolist()

    return {"query_vector": vector}

# def retrieve_node(state: RAGState, top_k: int = 2):
#     results = qdrant.query_points(
#         collection_name=COLLECTION_NAME,
#         query=state["query_vector"],   # ← pass vector directly
#         limit=top_k,
#         with_payload=True,
#     ).points                           # ← .points gives the list

#     chunks = []
#     for hit in results:
#         chunks.append({
#             "text": hit.payload.get("text"),
#             "heading": hit.payload.get("heading"),
#             "page_start": hit.payload.get("page_start"),
#             "score": round(hit.score, 4),
#         })

#     print(f"\n📄 Retrieved {len(chunks)} chunks:")
#     for i, c in enumerate(chunks, 1):
#         print(f"  {i}. [score={c['score']}] "
#               f"{c['heading'] or 'No heading'} | "
#               f"Page {c['page_start']} | "
#               f"{c['text'][:80]}...")

#     return {"retrieved_chunks": chunks}

def retrieve_node(state: RAGState, top_k: int = 10):

    # ─────────────────────────────────────────────
    # 1. Dense Retrieval from Qdrant
    # ─────────────────────────────────────────────
    dense_results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=state["query_vector"],
        limit=top_k,
        with_payload=True,
    ).points

    dense_chunks = []

    for hit in dense_results:
        dense_chunks.append({
            "id": str(hit.id),
            "text": hit.payload.get("text"),
            "heading": hit.payload.get("heading"),
            "page_start": hit.payload.get("page_start"),
            "dense_score": float(hit.score),
        })

    # ─────────────────────────────────────────────
    # 2. BM25 Sparse Retrieval
    # ─────────────────────────────────────────────

    # Fetch more documents for BM25
    all_docs = qdrant.scroll(
        collection_name=COLLECTION_NAME,
        with_payload=True,
        limit=200
    )[0]

    corpus = []
    corpus_chunks = []

    for point in all_docs:
        text = point.payload.get("text", "")

        corpus.append(preprocess(text))

        corpus_chunks.append({
            "id": str(point.id),
            "text": text,
            "heading": point.payload.get("heading"),
            "page_start": point.payload.get("page_start"),
        })

    bm25 = BM25Okapi(corpus)

    tokenized_query = state["question"].split()

    bm25_scores = bm25.get_scores(tokenized_query)

    sparse_chunks = []

    for chunk, score in zip(corpus_chunks, bm25_scores):

        sparse_chunks.append({
            "id": chunk["id"],
            "text": chunk["text"],
            "heading": chunk["heading"],
            "page_start": chunk["page_start"],
            "bm25_score": float(score),
        })

    # Sort BM25 results
    sparse_chunks = sorted(
        sparse_chunks,
        key=lambda x: x["bm25_score"],
        reverse=True
    )[:top_k]

    # ─────────────────────────────────────────────
    # 3. Hybrid Fusion
    # ─────────────────────────────────────────────

    fused_scores = defaultdict(float)
    chunk_lookup = {}

    # Dense weighting
    for rank, chunk in enumerate(dense_chunks):
        fused_scores[chunk["id"]] += 1 / (rank + 1)
        chunk_lookup[chunk["id"]] = chunk

    # Sparse weighting
    for rank, chunk in enumerate(sparse_chunks):
        fused_scores[chunk["id"]] += 1 / (rank + 1)
        chunk_lookup[chunk["id"]] = chunk

    # Final fused ranking
    ranked_ids = sorted(
        fused_scores,
        key=fused_scores.get,
        reverse=True
    )

    final_chunks = []

    for chunk_id in ranked_ids[:top_k]:

        chunk = chunk_lookup[chunk_id]

        chunk["hybrid_score"] = round(
            fused_scores[chunk_id],
            4
        )

        final_chunks.append(chunk)

    # ─────────────────────────────────────────────
    # 4. Logging
    # ─────────────────────────────────────────────

    print(f"\n📄 Hybrid Retrieved {len(final_chunks)} chunks:\n")

    for i, c in enumerate(final_chunks, 1):

        print(
            f"{i}. "
            f"[hybrid={c['hybrid_score']}] "
            f"{c.get('heading') or 'No heading'} | "
            f"Page {c.get('page_start')} | "
            f"{c['text'][:80]}..."
        )

    return {
        "retrieved_chunks": final_chunks
    }

def grade_docs_node(state: RAGState):
    scores = []
    for chunk in state["retrieved_chunks"]:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{
                "role": "user",
                "content": f"""
                    You are a retrieval evaluator.
                    A document is relevant if it contains information
                    that helps answer the question directly or partially.
                    Question: {state['question']}
                    Document: {chunk['text'][:400]}
                    Answer ONLY 'relevant' or 'irrelevant'."""
            }],
            temperature=0,
        )
        scores.append(response.choices[0].message.content.strip().lower())
    
    print(f"📊 Relevance scores: {scores}")
    return {"relevance_scores": scores}

def dense_retrieve_node(state: RAGState):

    results = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=state["query_vector"],
        limit=5,
        with_payload=True,
    ).points

    dense_chunks = []

    for hit in results:
        dense_chunks.append({
            "id": str(hit.id),
            "text": hit.payload.get("text"),
            "heading": hit.payload.get("heading"),
            "page_start": hit.payload.get("page_start"),
            "dense_score": float(hit.score),
        })

    return {"dense_chunks": dense_chunks}

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return text.split()

def sparse_retrieve_node(state: RAGState):

    all_docs = qdrant.scroll(
        collection_name=COLLECTION_NAME,
        with_payload=True,
        limit=200
    )[0]

    corpus = []
    corpus_chunks = []

    for point in all_docs:

        text = point.payload.get("text", "")

        corpus.append(text.split())

        corpus_chunks.append({
            "id": str(point.id),
            "text": text,
            "heading": point.payload.get("heading"),
            "page_start": point.payload.get("page_start"),
        })

    bm25 = BM25Okapi(corpus)

    scores = bm25.get_scores(
        state["question"].split()
    )

    sparse_chunks = []

    for chunk, score in zip(corpus_chunks, scores):

        sparse_chunks.append({
            "id": chunk["id"],
            "text": chunk["text"],
            "heading": chunk["heading"],
            "page_start": chunk["page_start"],
            "bm25_score": float(score)
        })

    sparse_chunks.sort(
        key=lambda x: x["bm25_score"],
        reverse=True
    )

    return {
        "sparse_chunks": sparse_chunks[:5]
    }
def fusion_node(state: RAGState):

    fused_scores = defaultdict(float)
    chunk_lookup = {}

    for rank, chunk in enumerate(state["dense_chunks"]):

        fused_scores[chunk["id"]] += 1 / (rank + 1)

        chunk_lookup[chunk["id"]] = chunk

    for rank, chunk in enumerate(state["sparse_chunks"]):
        fused_scores[chunk["id"]] += 1 / (rank + 1)

        chunk_lookup[chunk["id"]] = chunk

    ranked_ids = sorted(
        fused_scores,
        key=fused_scores.get,
        reverse=True
    )

    final_chunks = []

    for chunk_id in ranked_ids[:10]:

        chunk = chunk_lookup[chunk_id]

        chunk["hybrid_score"] = round(
            fused_scores[chunk_id],
            4
        )

        final_chunks.append(chunk)

    print("\n📄 Hybrid Retrieval Results\n")

    for i, chunk in enumerate(final_chunks, 1):

        print(
            f"{i}. "
            f"[{chunk['hybrid_score']}] "
            f"{chunk.get('heading') or 'No heading'}"
        )

    return {
        "retrieved_chunks": final_chunks
    }


reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)
def rerank_node(state: RAGState, top_k: int = 10):

    chunks = state.get("retrieved_chunks", [])

    if not chunks:
        return {"retrieved_chunks": []}

    pairs = [
        (state["question"], chunk["text"])
        for chunk in chunks
    ]

    scores = reranker.predict(
        pairs,
        batch_size=32,
        show_progress_bar=False
    )

    for chunk, score in zip(chunks, scores):
        chunk["rerank_score"] = float(score)

    reranked = sorted(
        chunks,
        key=lambda x: x["rerank_score"],
        reverse=True
    )[:top_k]

    return {
        "retrieved_chunks": reranked
    }

def rewrite_node(state: RAGState):
    if state["rewrite_count"] >= 2:          # hard limit on retries
        print("⚠️ Max rewrites reached, using original")
        return {"rewrite_count": state["rewrite_count"]}

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""Rewrite this question to improve document retrieval.
Original: {state['question']}
Return ONLY the rewritten question."""
        }],
        temperature=0.3,
    )
    new_q = response.choices[0].message.content.strip()
    new_vec = embed_model.encode(new_q).tolist()
    print(f"🔄 Rewritten: {new_q}")
    return {
        "question": new_q,
        "query_vector": new_vec,
        "rewrite_count": state["rewrite_count"] + 1
    }




def prompt_node(state: RAGState):
    context_parts = []

    for i, chunk in enumerate(state["retrieved_chunks"], 1):
        heading = f"[{chunk['heading']}]" if chunk["heading"] else ""
        page = f"(Page {chunk['page_start']})" if chunk["page_start"] else ""
        context_parts.append(
            f"Chunk {i} {heading} {page}:\n{chunk['text']}"
        )

    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know based on the document."

CONTEXT:
{context}

QUESTION:
{state['question']}

ANSWER:"""

    return {"prompt": prompt}




# ── Node 4: LLM response via Groq ────────────────────────────────
def llm_node(state: RAGState):
    print("\n🤖 Answer:\n")

    completion = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": "You are a document assistant. Answer only from the provided context."
            },
            {
                "role": "user",
                "content": state["prompt"]
            }
        ],
        temperature=0.2,
        top_p=1,
        reasoning_effort="medium",
        stream=True,
        stop=None
    )

    full_answer = ""
    for chunk in completion:
        token = chunk.choices[0].delta.content or ""
        print(token, end="", flush=True)
        full_answer += token

    print("\n")
    return {"answer": full_answer}



def hallucination_check_node(state: RAGState):
    context = "\n".join(c["text"][:300] for c in state["retrieved_chunks"])
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""Is this answer grounded in the context?
Context: {context}
Answer: {state['answer']}
Reply ONLY 'grounded' or 'hallucinated'."""
        }],
        temperature=0,
    )
    result = response.choices[0].message.content.strip().lower()
    print(f"🧠 Hallucination check: {result}")
    return {"hallucination_check": result}

def no_answer_node(state: RAGState):
    print("❌ Question is not relevant to this document.")
    return {"answer": "I don't know based on the document."}

def route_after_grading(state: RAGState):
    if "relevant" in state["relevance_scores"]:
        return "prompt"                    
    if state["rewrite_count"] >= 2:
        return "no_answer"                  

    return "rewrite"            

def route_after_hallucination(state: RAGState):
    if state["hallucination_check"] == "grounded":
        return END
    if state["rewrite_count"] >= 2:
        return END                  # give up gracefully
    return "prompt"                # retry generation

In [ ]:
# ── Build Graph ───────────────────────────────────────────────────
graph = StateGraph(RAGState)

graph.add_node("embed",    embed_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("grade",    grade_docs_node)       # NEW
graph.add_node("rewrite",  rewrite_node)           # NEW
graph.add_node("prompt",   prompt_node)
graph.add_node("llm",      llm_node)
graph.add_node("hallcheck", hallucination_check_node)  # NEW
graph.add_node("no_answer", no_answer_node)

graph.add_edge(START,       "embed")
graph.add_edge("embed",     "retrieve")
graph.add_edge("retrieve",  "grade")

# Branch after grading
graph.add_conditional_edges("grade", route_after_grading, {
    "prompt":  "prompt",
    "rewrite": "rewrite",
    "no_answer": "no_answer", 
})

# Rewrite loops back to embed (re-encode new question)
graph.add_edge("rewrite",   "embed")

graph.add_edge("prompt",    "llm")
graph.add_edge("llm",       "hallcheck")

# Branch after hallucination check
graph.add_conditional_edges("hallcheck", route_after_hallucination, {
    "prompt": "prompt",   # retry generation
    END: END,
})
graph.add_edge("no_answer", END) 
rag_app = graph.compile()

In [ ]:
result = rag_app.invoke({
        "question": "who is gandhi?",
        "query_vector": [],
        "retrieved_chunks": [],
        "prompt": "",
        "answer": "",
        "rewrite_count": 0 
    })


print(f"\n✅ Final Answer Stored:\n{result['answer']}")


In [ ]:
# Visualize graph
from IPython.display import Image, display
display(Image(rag_app.get_graph().draw_mermaid_png()))